In [1]:
# DAY 7 - AUTOMATED ADVANCE RETRIVAL
# AUTOMATIC PARENT CHILED --> META DATA FILTERING

import uuid
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

c:\Users\Sangram Mishra\Desktop\ADK\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
c:\Users\Sangram Mishra\Desktop\ADK\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 1. ORIGINAL DOCUMENT
# ============================================================

document = Document(
    page_content="""
    Producer Enterprise Financial Assistance

    Product planning receives 2 percent.

    Training receives 10 percent.

    Equipment and Machinery receives 32 percent.

    Software and IT receives 4 percent.

    Management receives 8 percent.

    Marketing activities are supported according
    to applicable guidelines.

    Maximum admissible assistance is ₹35 lakh.

    Producer Enterprises must comply with the
    applicable utilization and reporting guidelines.
    """,

    metadata={
        "scheme": "PE",
        "year": 2026,
        "district": "Kandhamal",
        "document_type": "Guideline",
        "source": "PE_Guidelines_2026.pdf"
    }
)

In [3]:
print(document)

page_content='
    Producer Enterprise Financial Assistance

    Product planning receives 2 percent.

    Training receives 10 percent.

    Equipment and Machinery receives 32 percent.

    Software and IT receives 4 percent.

    Management receives 8 percent.

    Marketing activities are supported according
    to applicable guidelines.

    Maximum admissible assistance is ₹35 lakh.

    Producer Enterprises must comply with the
    applicable utilization and reporting guidelines.
    ' metadata={'scheme': 'PE', 'year': 2026, 'district': 'Kandhamal', 'document_type': 'Guideline', 'source': 'PE_Guidelines_2026.pdf'}


In [4]:
#parent spiltter

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap= 100
)

In [6]:
#cretae parent chunks
parent_chunks = parent_splitter.split_documents(
    [document]
)

In [11]:
print(parent_chunks)
print(len(parent_chunks))

[Document(metadata={'scheme': 'PE', 'year': 2026, 'district': 'Kandhamal', 'document_type': 'Guideline', 'source': 'PE_Guidelines_2026.pdf'}, page_content='Producer Enterprise Financial Assistance\n\n    Product planning receives 2 percent.\n\n    Training receives 10 percent.\n\n    Equipment and Machinery receives 32 percent.\n\n    Software and IT receives 4 percent.\n\n    Management receives 8 percent.\n\n    Marketing activities are supported according\n    to applicable guidelines.\n\n    Maximum admissible assistance is ₹35 lakh.\n\n    Producer Enterprises must comply with the\n    applicable utilization and reporting guidelines.')]
1


In [10]:
for chunk in parent_chunks:
    print(type(chunk.page_content))

<class 'str'>


In [14]:
# now we will assighn the parent id to parent chunk
for parent in parent_chunks:
    parrent_id = str(uuid.uuid4())
    parent.metadata["parent_id"] = parrent_id

In [15]:
parent_chunks

[Document(metadata={'scheme': 'PE', 'year': 2026, 'district': 'Kandhamal', 'document_type': 'Guideline', 'source': 'PE_Guidelines_2026.pdf', 'parent_id': 'c8e13674-569c-4793-8cc4-e4add7853202'}, page_content='Producer Enterprise Financial Assistance\n\n    Product planning receives 2 percent.\n\n    Training receives 10 percent.\n\n    Equipment and Machinery receives 32 percent.\n\n    Software and IT receives 4 percent.\n\n    Management receives 8 percent.\n\n    Marketing activities are supported according\n    to applicable guidelines.\n\n    Maximum admissible assistance is ₹35 lakh.\n\n    Producer Enterprises must comply with the\n    applicable utilization and reporting guidelines.')]

In [16]:
# child splitter
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 250,
    chunk_overlap= 40
)

In [19]:
child_chunks = []
for parent in parent_chunks:
    children = child_splitter.split_documents([parent])
    for child in children:
        child.metadata["parent_id"] = (parent.metadata["parent_id"])
        child_chunks.append(child)


In [20]:
print(child_chunks)

[Document(metadata={'scheme': 'PE', 'year': 2026, 'district': 'Kandhamal', 'document_type': 'Guideline', 'source': 'PE_Guidelines_2026.pdf', 'parent_id': 'c8e13674-569c-4793-8cc4-e4add7853202'}, page_content='Producer Enterprise Financial Assistance\n\n    Product planning receives 2 percent.\n\n    Training receives 10 percent.\n\n    Equipment and Machinery receives 32 percent.\n\n    Software and IT receives 4 percent.\n\n    Management receives 8 percent.'), Document(metadata={'scheme': 'PE', 'year': 2026, 'district': 'Kandhamal', 'document_type': 'Guideline', 'source': 'PE_Guidelines_2026.pdf', 'parent_id': 'c8e13674-569c-4793-8cc4-e4add7853202'}, page_content='Management receives 8 percent.\n\n    Marketing activities are supported according\n    to applicable guidelines.\n\n    Maximum admissible assistance is ₹35 lakh.'), Document(metadata={'scheme': 'PE', 'year': 2026, 'district': 'Kandhamal', 'document_type': 'Guideline', 'source': 'PE_Guidelines_2026.pdf', 'parent_id': 'c8e1

In [21]:
## now we wil show the re;lationship between parent and child
for child in child_chunks:
    print("\nchild:")
    print(child.page_content)
    print(
        "parent_id",
        child.metadata["parent_id"]
    )


child:
Producer Enterprise Financial Assistance

    Product planning receives 2 percent.

    Training receives 10 percent.

    Equipment and Machinery receives 32 percent.

    Software and IT receives 4 percent.

    Management receives 8 percent.
parent_id c8e13674-569c-4793-8cc4-e4add7853202

child:
Management receives 8 percent.

    Marketing activities are supported according
    to applicable guidelines.

    Maximum admissible assistance is ₹35 lakh.
parent_id c8e13674-569c-4793-8cc4-e4add7853202

child:
Producer Enterprises must comply with the
    applicable utilization and reporting guidelines.
parent_id c8e13674-569c-4793-8cc4-e4add7853202


In [24]:
#Embedding model
embedings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1659.67it/s]


In [27]:
#vector database
vectorstores = Chroma.from_documents(
    documents = child_chunks,
    embedding = embedings,
    collection_name = "ormas_parent_child"
)

In [29]:
#build paren store
parent_store = {}
for parent in parent_chunks:
    parent_id = parent.metadata["parent_id"]
    parent_store[parent_id] = parent
print(parent_store)

{'c8e13674-569c-4793-8cc4-e4add7853202': Document(metadata={'scheme': 'PE', 'year': 2026, 'district': 'Kandhamal', 'document_type': 'Guideline', 'source': 'PE_Guidelines_2026.pdf', 'parent_id': 'c8e13674-569c-4793-8cc4-e4add7853202'}, page_content='Producer Enterprise Financial Assistance\n\n    Product planning receives 2 percent.\n\n    Training receives 10 percent.\n\n    Equipment and Machinery receives 32 percent.\n\n    Software and IT receives 4 percent.\n\n    Management receives 8 percent.\n\n    Marketing activities are supported according\n    to applicable guidelines.\n\n    Maximum admissible assistance is ₹35 lakh.\n\n    Producer Enterprises must comply with the\n    applicable utilization and reporting guidelines.')}


In [36]:
#filttered child search
def retrive_children(
    query,
    scheme = None,
    year = None,
    district = None,
    k = 5):
    filter = {}
    if scheme is not None:
        filter["scheme"] = scheme
    if year is not None:
        filter["year"] = year
    if district is not None:
        filter["district"] = district

    # For a single-condition demonstration,
    # use the appropriate Chroma filter.
    #
    # For multiple conditions, production code
    # should construct the backend-specific
    # logical filter explicitly.
    if len(filter) == 1:
        results = vectorstores.similarity_search(
            query,
            k=k,
            filter=filter
        )
    else:
         # Educational fallback:
        # retrieve candidates first and filter
        # metadata in Python.

        results = vectorstores.similarity_search(
            query,
            k=k * 3
        )
        for key , value in filter.items():
            results = [doc for doc in results if doc.metadata.get(key) == value]
    return results[:k]



In [37]:
vectorstores.similarity_search(
            "what is financial assistance of pe",
            k=3)

[Document(id='ecb403dc-3d2f-49bf-a82f-2a30e3e8bab7', metadata={'district': 'Kandhamal', 'parent_id': 'c8e13674-569c-4793-8cc4-e4add7853202', 'scheme': 'PE', 'year': 2026, 'document_type': 'Guideline', 'source': 'PE_Guidelines_2026.pdf'}, page_content='Producer Enterprise Financial Assistance\n\n    Product planning receives 2 percent.\n\n    Training receives 10 percent.\n\n    Equipment and Machinery receives 32 percent.\n\n    Software and IT receives 4 percent.\n\n    Management receives 8 percent.'),
 Document(id='ab528066-ae05-4d7d-895c-7e9625e78ca6', metadata={'document_type': 'Guideline', 'parent_id': 'c8e13674-569c-4793-8cc4-e4add7853202', 'source': 'PE_Guidelines_2026.pdf', 'year': 2026, 'district': 'Kandhamal', 'scheme': 'PE'}, page_content='Management receives 8 percent.\n\n    Marketing activities are supported according\n    to applicable guidelines.\n\n    Maximum admissible assistance is ₹35 lakh.'),
 Document(id='3d5b39a1-e440-491a-990e-b7f0f1306285', metadata={'parent_

In [46]:
# parent retrival

def retrive_parent(children):
    parent = []
    seen = set()
    for child in children:
        parent_id = child.metadata.get("parent_id")
        if parent_id in parent_store and parent_id not in seen:
            parent.append(parent_store[parrent_id])
            seen.add(parent_id)
    return parent

In [47]:
# ============================================================
# 13. COMPLETE PARENT-CHILD RETRIEVAL
# ============================================================

def parent_child_search(
    query,
    scheme=None,
    year=None,
    district=None
):

    # --------------------------------------------
    # STEP 1 — Filtered child retrieval
    # --------------------------------------------

    children = retrive_children(
        query=query,
        scheme=scheme,
        year=year,
        district=district,
        k=5
    )


    # --------------------------------------------
    # STEP 2 — Recover parent documents
    # --------------------------------------------

    parents = retrive_parent(
        children
    )


    return {
        "children": children,
        "parents": parents
    }


In [ ]:
# ============================================================
# lets test the results

result = parent_child_search(
    query="What percentage is allocated to equipment?",
    scheme="PE",
    year=2026,
    district="Kandhamal"
)


In [50]:
# ============================================================
# 15. DISPLAY RESULTS
# ============================================================

print("\n================ CHILDREN ================\n")

for child in result["children"]:

    print(
        child.page_content
    )

    print(
        "Parent:",
        child.metadata["parent_id"]
    )

    print("-" * 50)


print("\n================ PARENTS ================\n")

for parent in result["parents"]:

    print(
        parent.page_content
    )

    print("-" * 50)


================ CHILDREN ================

Producer Enterprise Financial Assistance

    Product planning receives 2 percent.

    Training receives 10 percent.

    Equipment and Machinery receives 32 percent.

    Software and IT receives 4 percent.

    Management receives 8 percent.
Parent: c8e13674-569c-4793-8cc4-e4add7853202
--------------------------------------------------
Management receives 8 percent.

    Marketing activities are supported according
    to applicable guidelines.

    Maximum admissible assistance is ₹35 lakh.
Parent: c8e13674-569c-4793-8cc4-e4add7853202
--------------------------------------------------
Producer Enterprises must comply with the
    applicable utilization and reporting guidelines.
Parent: c8e13674-569c-4793-8cc4-e4add7853202
--------------------------------------------------

================ PARENTS ================

Producer Enterprise Financial Assistance

    Product planning receives 2 percent.

    Training receives 10 percent.

    E